# Сравнение сочетаний функций при изменении назначения территории

Ноутбук считает метод от исходных слоёв до таблицы сравнения вариантов. Вручную ничего считать не нужно.

Расчёт идёт по **правилам v5** (задание, раздел 4): рабочие места — по категориям зданий и нормам площади на работника по типу сервиса; варианты — пять сочетаний функций; КИТ варианта — из ПЗЗ или по СП 42.13330.2016 (табл. Б.1). Правила v4 (прежний ноутбук) остались для сверки: примеры `spb`, `gatchina`, `vysotsk_v4`.

**Порядок работы**
1. Положите свои слои в папку `data` рядом с ноутбуком.
2. Заполните анкету в ячейке **«Шаг 0. Ваши данные»**: пути к файлам и имена полей.
3. Выполните все ячейки (**Run All**). Ячейка **«Шаг 0б. Проверка данных»** покажет, что найдено в слоях. Если есть ошибки (✕), расчёт остановится и подскажет, что исправить.
4. Результаты появятся в папке `results`.

**Какие слои нужны**

| Слой | Геометрия | Какие поля нужны |
|---|---|---|
| Кварталы | полигоны | номер квартала (можно без него — пронумеруются по порядку) |
| Здания | полигоны | жилое или нет; этажность; по желанию — площадь застройки, жилая площадь, жители |
| Сервисы | точки или полигоны | тип объекта; у школ и детских садов — число мест |
| Зоны (по желанию) | полигоны | код зоны ПЗЗ или функциональной зоны (или поле зоны в слое кварталов) |
| Предприятия (по желанию) | точки | тип сервиса группы П: точки предприятий в производственных зонах — без них одноэтажный цех меньше 1 000 м² считается хозпостройкой |

Тип сервиса относится к группе по файлу **`таблица_групп_сервисов.csv`** (открывается в Excel). Правило таблицы — код ВРИ объекта:
- **К** — 3.3, 4.2, 4.3, 4.4, 4.6;
- **ОД** — 3.2.3, 3.5.2, 3.6.1, 3.8, 4.1, 4.5, 4.7;
- **П** — 6.0–6.12;
- школы и детские сады учитываются по числу мест, метро и остановки — по расстоянию до территории;
- остальные типы — группа «нет», в расчёте не участвуют.

Если в ваших данных есть тип, которого нет в таблице, проверка покажет его. Тогда добавьте строку: тип, код ВРИ, группа.

**Что нужно установить.** Python 3.10+ и библиотеки numpy, pandas, matplotlib, pyyaml. Все они есть в Anaconda и Google Colab.

**Расчёт выполняет пакет `mixuse`** — папка `mixuse/` должна лежать рядом с ноутбуком (в Google Colab загрузите её вместе с ноутбуком). Ноутбук заполняет анкету, вызывает функции пакета по шагам и показывает результаты. Те же расчёты выполняет командная строка: `python -m mixuse compare configs/<код>.yaml`.

## Шаг 0. Ваши данные

Заполните анкету: впишите пути к файлам и имена полей **так, как они называются в вашем слое** (их видно в QGIS: «Свойства слоя» → «Поля»). Если поля нет, оставьте `None`.

- Пути пишутся от папки, где лежит ноутбук.
- Форматы файлов: GeoJSON (`.geojson`) или GeoPackage (`.gpkg`).
- Все слои должны быть в одной системе координат: EPSG:4326 (градусы) или метрической (UTM, МСК). Web Mercator (EPSG:3857) не подходит.

**Сервисы** можно дать двумя способами:
- **папкой**, где один файл — один тип, а имя файла и есть тип: `school.geojson`, `cafe.geojson`… Так устроены данные blocksnet;
- **одним файлом** с полем типа. Для выгрузки из OpenStreetMap укажите список полей тегов, например `['amenity', 'shop', 'office', 'tourism']`. Тип тогда записывается как тег: `amenity=school`. Такие теги указаны в столбце «тег_OSM» таблицы групп.

In [ ]:
# ===== ШАГ 0. ВАШИ ДАННЫЕ — заполняется только эта ячейка =====
ДАННЫЕ = dict(
    название='Мой город',
    код='moy_gorod',                  # короткое имя для файлов результатов (латиницей, без пробелов)
    правила='v5',                     # правила расчёта: 'v5' — раздел 4 задания; 'v4' — прежний ноутбук (для сверки)
    режим='слои',                     # 'слои' — кварталы, здания и сервисы отдельными слоями
                                      # 'таблица' — к кварталам уже присоединены здания и сервисы (см. пример Гатчины)
    кварталы=dict(
        файл='data/moy_gorod/кварталы.geojson',
        поле_номера='block_id',       # поле с номером квартала; None — пронумеровать по порядку
    ),
    здания=dict(
        файл='data/moy_gorod/здания.geojson',
        поле_жилое='is_living',       # жилое ли здание: 1/0, True/False, да/нет
        поле_этажность='storeys',     # число этажей; если пусто — 1 этаж
        поле_площадь_застройки=None,  # None — по контуру здания
        поле_жилая_площадь=None,      # None — поэтажная площадь × k_liv
        поле_жители=None,             # None — жилая площадь ÷ q
    ),
    сервисы=dict(
        файл='data/moy_gorod/сервисы',   # папка (один файл = один тип), один файл или список: ['…/сервисы', '…/промышленность']
        поле_типа=None,               # для одного файла: 'type' или список полей OSM ['amenity', 'shop', 'office', 'tourism']
        поле_мест='capacity',         # число мест у школ и детских садов
    ),
    зоны=dict(                        # необязательно: если зон нет, оставьте файл=None
        файл=None,
        поле_кода=None,
        поле_в_кварталах=None,        # если зона записана в слое кварталов (как в Гатчине): имя этого поля
        правила=[('ТП', 'П'), ('ТД', 'ОД')],   # код зоны начинается с «ТП» → производственная, «ТД» → общественно-деловая
    ),
    территория=[],                    # номера кварталов, где меняется назначение, например [322, 323, 377]
)
ТАБЛИЦА_ГРУПП = 'таблица_групп_сервисов.csv'

### Примеры заполнения

Так анкета заполнена для наборов данных, на которых проверялся метод. Чтобы посчитать пример, поставьте, например, `ПРИМЕР = 'vysotsk'`. Для своих данных — `ПРИМЕР = None`.

Правила v5:
- **spb_full** — Санкт-Петербург целиком (`data/spb_full/`, около 150 МБ; в репозиторий не входит). КИТ 1,4 — значение пользователя (ПЗЗ Санкт-Петербурга).
- **gatchina_layers** — Гатчина послойно: кварталы, здания, сервисы и 12 предприятий по отметке автора; зона — поле `land_use` слоя кварталов.
- **vysotsk** — Высоцк: кварталы, здания, сервисы (две папки) и зоны генплана.

Правила v4 (прежний ноутбук, для сверки с эталонами): **spb** — Васильевский остров, **gatchina** — Гатчина готовой таблицей, **vysotsk_v4** — Высоцк.

In [ ]:
ПРИМЕР = None    # None — ваши данные из шага 0; пример: 'spb_full', 'gatchina_layers', 'vysotsk' (v5) или 'spb', 'gatchina', 'vysotsk_v4' (v4)

ВЫСОЦК = dict(
    кварталы=dict(файл='examples/vysotsk/кварталы.geojson', поле_номера=None),   # кварталы пересохранены в EPSG:4326, номер — по порядку
    здания=dict(файл='examples/vysotsk/здания.geojson', поле_жилое='is_living', поле_этажность='number_of_floors',
                поле_площадь_застройки='footprint_area', поле_жилая_площадь='living_area', поле_жители='population'),
    сервисы=dict(файл=['examples/vysotsk/сервисы', 'examples/vysotsk/промышленность'], поле_типа=None, поле_мест='capacity'),
    зоны=dict(файл='examples/vysotsk/зоны.geojson', поле_кода='Зона', правила=[('Производственная', 'П'), ('Общественно-деловая', 'ОД')]),
    территория=[362, 387, 392],             # производственные кварталы у жилой застройки (пример)
)

ПРИМЕРЫ = {
    # ---------- правила v5
    'spb_full': dict(
        название='Санкт-Петербург', код='spb_full', правила='v5', режим='слои',
        кварталы=dict(файл='data/spb_full/blocks.geojson', поле_номера=None),   # номер = порядок в файле (задание, 4.1.3)
        здания=dict(файл='data/spb_full/buildings.geojson', поле_жилое='is_living', поле_этажность='storeys_count',
                    поле_площадь_застройки='building_area', поле_жилая_площадь='living_area', поле_жители='population_balanced'),
        сервисы=dict(файл='data/spb_full/services', поле_типа=None, поле_мест='capacity'),
        зоны=dict(файл='data/spb_full/functional_zones.geojson', поле_кода='functional_zone', правила=[('ТП', 'П'), ('ТД', 'ОД')]),
        территория=[8346, 9366, 2059],          # промзона у наб. Макарова
        параметры=dict(kit=1.4, kit_source='ПЗЗ Санкт-Петербурга, зоны ТСМ и ТД1-1', stat_jobs=1472900),   # КИТ из ПЗЗ
        источники=dict(stat_jobs='Петростат, 2022 — работники организаций без малого бизнеса'),
    ),
    'gatchina_layers': dict(
        название='Гатчина', код='gatchina_layers', правила='v5', режим='слои',
        кварталы=dict(файл='data/gatchina_layers/кварталы.gpkg', поле_номера='block_id'),
        здания=dict(файл='data/gatchina_layers/здания.geojson', поле_жилое='is_living', поле_этажность='number_of_floors',
                    поле_площадь_застройки='footprint_area', поле_жилая_площадь='living_area', поле_жители='population'),
        сервисы=dict(файл='data/gatchina_layers/services', поле_типа=None, поле_мест='capacity'),
        зоны=dict(файл=None, поле_в_кварталах='land_use', правила=[('LandUse.INDUSTRIAL', 'П'), ('LandUse.BUSINESS', 'ОД')]),
        территория=[322, 323, 377],             # утверждена автором 19.11.2026
        параметры=dict(kit_sp_row='zh_low_mid'),   # КИТ в ПЗЗ Гатчины не установлен → СП 42.13330.2016, табл. Б.1
        источники=dict(kit_sp_row='до решения автора — [НУЖНЫ ДАННЫЕ: зона ПЗЗ, в которую попадает территория]'),
    ),
    'vysotsk': dict(название='Высоцк', код='vysotsk', правила='v5', режим='слои', **ВЫСОЦК,
                    параметры=dict(kit_sp_row='zh_low_mid'),
                    источники=dict(kit_sp_row='до решения автора — [НУЖНЫ ДАННЫЕ: зона ПЗЗ, в которую попадает территория]')),
    # ---------- правила v4 (прежний ноутбук, для сверки с эталонами)
    'spb': dict(
        название='Санкт-Петербург, Васильевский остров', код='spb', правила='v4', режим='слои',
        кварталы=dict(файл='examples/spb_vasilievsky/blocks.geojson', поле_номера='block_id'),
        здания=dict(файл='examples/spb_vasilievsky/buildings.geojson', поле_жилое='is_living', поле_этажность='storeys_count',
                    поле_площадь_застройки='building_area', поле_жилая_площадь='living_area', поле_жители='population_balanced'),
        сервисы=dict(файл='examples/spb_vasilievsky/services', поле_типа=None, поле_мест='capacity'),
        зоны=dict(файл='examples/spb_vasilievsky/functional_zones.geojson', поле_кода='functional_zone', правила=[('ТП', 'П'), ('ТД', 'ОД')]),
        территория=[8346, 9366, 2059],
    ),
    'gatchina': dict(
        название='Гатчина', код='gatchina', правила='v4', режим='таблица',
        кварталы=dict(файл='examples/gatchina/blocks_with_services.gpkg', поле_номера='block_id'),
        готовая_таблица=dict(поле_жители='population', поле_жилая_площадь='living_area', поле_нежилая_площадь='non_living_area',
                             поле_зоны='land_use', префикс_числа='count_', префикс_мест='capacity_',
                             правила=[('LandUse.INDUSTRIAL', 'П'), ('LandUse.BUSINESS', 'ОД')]),
        территория=[322, 323, 377],
    ),
    'vysotsk_v4': dict(название='Высоцк', код='vysotsk', правила='v4', режим='слои', **ВЫСОЦК),
}

## Параметры метода и варианты

- `P` — параметры правил v5. Значения — по умолчанию из задания (раздел 5); чтобы задать своё, измените число. Программа покажет у каждого параметра, откуда взято значение: «пользователь», «по данным» или «по умолчанию».
- **КИТ варианта** (задание, 4.6.2) — один для всех вариантов: `kit` — КИТ из ПЗЗ или проекта планировки (источник — `kit_source`); если в ПЗЗ его нет — `kit_sp_row`: строка таблицы Б.1 СП 42.13330.2016 по типу зоны, в которую переводится территория (предельное значение — вариант показывает наибольший допустимый объём). `pzz_build_share` × `pzz_max_storeys` — предел по ПЗЗ, только проверка. КИТ окружения по данным выводится лишь для сравнения.
- `SCN` — варианты сочетаний: доли поэтажной площади в процентах (задание, 4.6.1).
- Для примеров v4 параметры и варианты этой ячейки не применяются: берутся значения прежнего ноутбука.

In [ ]:
P = dict(
    kit=None,           # КИТ из ПЗЗ или проекта планировки, например 1.4
    kit_source=None,    # его источник, например 'ПЗЗ Санкт-Петербурга, зоны ТСМ и ТД1-1'
    kit_sp_row=None,    # если КИТ в ПЗЗ нет — строка табл. Б.1 СП 42.13330.2016: 'zh_multi' 1,2; 'zh_multi_recon' 1,6;
                        #   'zh_low_mid' 0,8; 'zh_block' 0,6; 'zh_individual' 0,4; 'od_multi' 3,0; 'od_special' 2,4;
                        #   'p_industrial' 2,4; 'p_science' 1,0; 'p_storage' 1,8
    pzz_build_share=None, pzz_max_storeys=None,   # проверка: максимальный процент застройки × этажность по ПЗЗ
    R=800,              # радиус окружения, м
    r=500,              # радиус пешей доступности функции, м
    q=30,               # м² жилой площади на человека
    k_liv=0.7,          # доля жилой площади в поэтажной
    a_К=28, a_ОД=15, a_П=69,   # м² на работника в вариантах (и для типа сервиса без нормы): магазины и кафе, офисы, производство
    n_school=110, n_kinder=60, # мест в школах и детских садах на 1000 жителей
    S_min=200,          # нежилые здания меньше этой площади застройки не учитываются, м²
    storeys_default=1,  # этажность, если она не указана
    builtin=True,       # учитывать встроенные помещения: сервисы К и ОД внутри контура жилого дома
    builtin_s0=100,     # площадь одного встроенного помещения, м² (допущение; проверка 50 и 300)
    builtin_share=0.5,  # встроенные помещения занимают не больше этой доли первого этажа
    builtin_min_storeys=2,              # правило применяется к жилым домам не ниже стольких этажей
    round_threshold=0.88, round_min_vertices=10,   # круглый контур без сервиса в производственной зоне — сооружение
    prod_min_storeys=2,                 # здание производственной зоны без сервиса от 2 этажей — A ÷ a_prod_zone
    prod_min_footprint_one_storey=1000, # одноэтажное здание производственной зоны без сервиса от этой площади — A ÷ a_prod_one_storey
    a_prod_zone=69, a_prod_one_storey=81, a_business_zone=50,   # м² на работника: производственная зона, цех или склад, общественно-деловая зона
    social_norms=dict(school=13.5, kindergarten=35, hospital=20, roddom=20, polyclinic=32, dentist=32,
                      trauma=32, women_clinic=32, emergency=32),   # школа — учеников на работника; остальные — м² на работника
    stat_jobs=None,     # число работников по статистике для проверки 4.8.4 (если есть)
)
SCN = {   # доли поэтажной площади, %: Ж — жильё, К — торговля и услуги, ОД — общественно-деловая, П — производство
    'Ж+К':    {'Ж': 85, 'К': 15, 'ОД': 0,  'П': 0},
    'Ж+К+ОД': {'Ж': 75, 'К': 5,  'ОД': 20, 'П': 0},
    'К+ОД':   {'Ж': 0,  'К': 15, 'ОД': 85, 'П': 0},
    'ОД+П':   {'Ж': 0,  'К': 10, 'ОД': 40, 'П': 50},
    'Ж+П':    {'Ж': 50, 'К': 10, 'ОД': 0,  'П': 40},
}
OUT = 'results'

## Функции метода — пакет `mixuse`

Чтение слоёв, перевод градусов в метры, проверка «точка внутри полигона», расстояния и формулы шагов 1–9 находятся в пакете `mixuse` (папка рядом с ноутбуком). Эту ячейку менять не нужно.

| Модуль | Что делает |
|---|---|
| `mixuse.geometry` | чтение GeoJSON и GeoPackage, перевод в метры, площадь, точка в полигоне, расстояния |
| `mixuse.groups` | таблица групп сервисов |
| `mixuse.blocks` | шаги 1–4: таблица кварталов (`load_layers`, `build_table`) |
| `mixuse.compare` | шаги 5–6: окружение и сравнение вариантов (`compare`) |
| `mixuse.checks` | проверка данных; шаги 7 и 7б (`sensitivity`, `builtin_sensitivity`) |
| `mixuse.export` | таблицы CSV, слой для QGIS, карта, файл для страницы |
| `mixuse.notebook` | перевод анкеты шага 0 в настройки пакета, печать проверки |

In [ ]:
# ===== ФУНКЦИИ МЕТОДА — из пакета mixuse (менять не нужно) =====
import os, time
import pandas as pd
import matplotlib.pyplot as plt
from mixuse.notebook import config_from_form, print_check
from mixuse.blocks import load_layers, build_table
from mixuse.compare import compare
from mixuse.checks import sensitivity, builtin_sensitivity, estimates_check, stats_check, threshold_check
from mixuse import export

## Шаг 0б. Проверка данных

Ячейка открывает слои и печатает, что в них нашлось:
- ✓ — всё в порядке;
- ! — предупреждение: расчёт пойдёт, но результат стоит проверить;
- ✕ — ошибка: расчёт остановится.

Проверяется:
- есть ли файлы и поля из анкеты;
- уникальны ли номера кварталов и есть ли кварталы территории;
- сколько жилых зданий и у скольких нет этажности;
- какие типы сервисов попали в группы К, ОД, П;
- какие типы не найдены в таблице групп;
- есть ли число мест у школ и детских садов;
- одна ли система координат у всех слоёв.

In [ ]:
Д = ПРИМЕРЫ[ПРИМЕР] if ПРИМЕР else ДАННЫЕ
cfg = config_from_form(Д, P, SCN, ТАБЛИЦА_ГРУПП)          # анкета → настройки пакета
P, SCN = cfg['params'], cfg['scenarios']                  # параметры и варианты, проверенные пакетом
V5 = cfg['rules'] == 'v5'
print(f'Набор данных: {cfg["title"]} (правила {cfg["rules"]}, режим «{Д.get("режим", "слои")}»)\n')
print_check(cfg)

## Шаги 1–4. Таблица кварталов

1. **Слои переводятся в метры.**
2. **Каждый сервис получает группу** по таблице групп.
3. **Каждое здание и сервис сводится к точке** внутри него и относится к кварталу, где лежит эта точка.
4. **По каждому зданию и кварталу считаются** (правила v5, задание, раздел 4):
    - **поэтажная площадь** A = площадь застройки × этажность; нет этажности — 1 этаж;
    - **жители** — из поля; пусто у жилого дома — жилая площадь ÷ q (если так добавится больше 1 % жителей, расчёт остановится);
    - **встроенные помещения** жилого дома не ниже 2 этажей с сервисами К и ОД: B = min(n · 100 м²; 0,5 · S), рабочие места — B ÷ норма типа сервиса;
    - **рабочие места нежилого здания** (от 200 м²) — первое выполненное условие: сервис К, ОД, П в контуре → A ÷ норма типа; социальный объект → норма объекта (школа — мест ÷ 13,5); в производственной зоне: круглый контур — сооружение (0), от 2 этажей — A ÷ 69, одноэтажное от 1 000 м² — A ÷ 81, меньше — 0; в общественно-деловой зоне — A ÷ 50; иначе 0;
    - **две оценки**: основная (А + Б) и нижняя граница (только А — функция подтверждена сервисом);
    - **есть ли функция**: К, ОД — есть сервис или встроенное помещение группы; П — есть сервис П или здание производственной зоны, давшее рабочие места;
    - **места в школах и детских садах** — сумма мест.

Таблица зданий `results/<код>_здания.csv` (только v5) показывает шаг, категорию и рабочие места каждого здания — её можно открыть в QGIS по полям x и y.

In [ ]:
os.makedirs(OUT, exist_ok=True)
КОД = cfg['name']; t0 = time.time()
L = load_layers(cfg, cfg['_groups'])          # слои, приведённые к системе координат кварталов
city = build_table(L, cfg, P, cfg['_groups'])
print(f'Подготовка заняла {time.time() - t0:.0f} с')
for note in city.get('crs_notes', []):
    print(f'  пересчёт координат: {note}')
for k, v in city['report'].items():
    print(f'  {k}: {v}')
export.blocks_csv(city, f'{OUT}/{КОД}_таблица_кварталов.csv')
if V5:
    export.buildings_csv(city, f'{OUT}/{КОД}_здания.csv')
tab = city['table']
tab[['block_id', 'zone', 'area', 'pop', 'jobs', 'есть_К', 'есть_ОД', 'есть_П', 'cap_school', 'cap_kinder']].round(0).head(8)

## Шаги 5–6. Окружение и сравнение вариантов

5. **Окружение** — кварталы, центр которых не дальше R от границы территории.
6. **КИТ варианта** (один для всех вариантов): из ПЗЗ или проекта планировки (`kit`); если его нет — по СП 42.13330.2016, табл. Б.1 (`kit_sp_row`). КИТ окружения по данным — только для сравнения: строка «КИТ варианта / окружения (в N раз плотнее)».
7. **Для каждого варианта** существующая застройка территории заменяется полностью:
    - F = площадь территории × КИТ; новые жители N = F × доля жилья × k_liv ÷ q; рабочие места J = F × (доля К ÷ 28 + доля ОД ÷ 15 + доля П ÷ 69);
    - **рабочих мест на жителя в окружении** K = (рабочие места окружения + J) ÷ (жители окружения + N);
    - **доступ к торговле и общественно-деловой функции** — сколько жителей окружения получают или теряют его в радиусе r; **производство** — сколько жителей живут в радиусе r от него до и после;
    - **школы и детские сады**: дети с территории и сколько из них без свободного места в окружении (свободные места = места окружения − потребность его жителей).

**Краткая таблица** (7 строк и вывод) — основной результат; **подробная** сохраняется в `results/<код>_сравнение_подробно.csv`. Сводного балла и «лучшего варианта» нет: вывод описывает последствия каждого варианта, выбор делает человек.

In [ ]:
T = cfg['territory']
table, info, det = compare(city, T, P, SCN)
for k, v in info.items():
    if k != 'предупреждения':
        print(f'{k}: {v}')
for w in info.get('предупреждения', []):
    print('  ! ' + w)
if V5:
    export.table_csv(det['short'], f'{OUT}/{КОД}_сравнение_вариантов.csv')
    export.table_csv(table, f'{OUT}/{КОД}_сравнение_подробно.csv')
    print('\nВывод:')
    for name, text in det['conclusions'].items():
        print(f'  {name}: {text}')
    display(det['short']) if 'display' in globals() else print(det['short'].to_string())
else:
    export.table_csv(table, f'{OUT}/{КОД}_сравнение_вариантов.csv')
    display(table) if 'display' in globals() else print(table.to_string())

## Шаг 7. Проверка устойчивости

Расчёт повторяется при других радиусах окружения и КИТ. Для каждого сочетания показано, на сколько вариант меняет K окружения относительно «как есть». Если знак изменения сохраняется, а величина меняется мало, вывод не зависит от выбранных значений. Для школ показано, сколько детей с территории остаются без свободного места в окружении (v4 — сколько мест не хватает).

In [ ]:
stab = sensitivity(city, T, P, SCN)          # R = 500, 800, 1000 м × КИТ = 1,4 и 2,0
export.table_csv(stab, f'{OUT}/{КОД}_устойчивость.csv', index=False)
stab

## Шаг 7б. Проверка допущения о встроенных помещениях

Площадь одного встроенного помещения `builtin_s0` — допущение. Таблица кварталов и сравнение вариантов пересчитываются без встроенных помещений и при s₀ = 50, 100, 300 м². Если K окружения меняется мало, вывод от этого допущения не зависит. Только для режима «слои».


In [ ]:
bi = builtin_sensitivity(L, cfg, P, T, SCN)   # без встроенных и s0 = 50, 100, 300 м²
if bi is not None:
    export.table_csv(bi, f'{OUT}/{КОД}_встроенные.csv', index=False)
    display(bi) if 'display' in globals() else print(bi.to_string(index=False))
else:
    print('Режим «таблица»: слоя зданий нет, встроенные помещения не учитываются.')

## Проверки 4.8.3–4.8.5 (только правила v5)

- **4.8.3** — основная оценка рабочих мест (А + Б) против нижней границы (только А): K окружения в обеих оценках.
- **4.8.4** — сравнение со статистикой, если задан `stat_jobs`.
- **4.8.5** — порог площади одноэтажного здания производственной зоны: 500 / 1 000 / 2 000 м².

In [ ]:
if V5:
    for title, df, suffix in [('4.8.3. Основная оценка и нижняя граница', estimates_check(city, T, P, SCN), 'оценки'),
                              ('4.8.4. Сравнение со статистикой', stats_check(city, P), 'статистика'),
                              ('4.8.5. Порог площади одноэтажного здания', threshold_check(L, cfg, P, T, SCN), 'порог_цеха')]:
        if df is not None:
            export.table_csv(df, f'{OUT}/{КОД}_{suffix}.csv', index=False)
            print(title); display(df) if 'display' in globals() else print(df.to_string(index=False))
else:
    print('Правила v4: проверок 4.8.3–4.8.5 нет.')

## Шаг 8. Карта и слой для QGIS

На карте: территория — фиолетовая, окружение обведено. Для выбранного варианта жилые кварталы окрашены по изменению доступа: зелёный — функции появились, оранжевый — пропали. Тот же результат сохраняется в слой `results/<город>_кварталы.geojson`. Его можно открыть в QGIS и оформить для рисунков НИР.

In [ ]:
SHOW = list(SCN)[1]   # какой вариант показать на карте (по умолчанию второй: Ж+К+ОД, в v4 — С1 Ж+ОД)

fig = export.map_figure(city, det, SHOW, P['R'], cfg['title'])
fig.savefig(f'{OUT}/{КОД}_карта_{SHOW.split()[0]}.png', dpi=150, bbox_inches='tight')
plt.show()

export.blocks_geojson(city, f'{OUT}/{КОД}_кварталы.geojson', det, SCN)
print('Слой для QGIS сохранён:', f'{OUT}/{КОД}_кварталы.geojson')

## Шаг 9. Файл данных для веб-страницы

Ноутбук — «движок»: он готовит таблицу кварталов из исходных слоёв. Веб-страница — «витрина»: она загружает этот файл и при каждом выборе территории сама пересчитывает шаги 5–6 по тем же формулам. Файл сохраняется в формате, который читает страница «Конструктор смешанности»: кварталы с контурами и показателями, точки метро и остановок, параметры, варианты и территория из шага 0.

In [ ]:
if V5:
    print('Файл для страницы по правилам v5 появится на этапе 7 (задание, раздел 10).')
else:
    export.web_json(city, cfg, f'{OUT}/{КОД}_для_страницы.json')
    print('Файл для страницы:', f'{OUT}/{КОД}_для_страницы.json', '—', len(city['table']), 'кварталов')

## Шаг 10. Перебор сочетаний

Программа перебирает все сочетания долей допустимых групп с заданным шагом (в сумме 100 %) и для каждого считает те же показатели, что для вариантов. Сводка показывает диапазон каждого показателя и то, доля какой группы на него влияет сильнее всего (коэффициент корреляции r и изменение показателя при доле группы больше на 10 п. п.). **Сочетания не сортируются «от лучшего к худшему»**, рекомендаций нет; можно отфильтровать сочетания своими условиями.

- `ГРУППЫ` — допустимые группы; `ШАГ` — шаг долей, %; `МАКС`, `МИН` — пределы долей (по умолчанию К не больше 20 %);
- `УСЛОВИЯ` — фильтр, например `['K окружения>=0.5', 'Жители на территории N<=5000']`.

In [ ]:
ГРУППЫ = ['Ж', 'К', 'ОД']
ШАГ = 10
МАКС = {'К': 20}
МИН = {}
УСЛОВИЯ = []

if V5:
    from mixuse.scenarios import enumerate_shares, explore, apply_filters, explore_summary
    комбинации = enumerate_shares(ГРУППЫ, ШАГ, МАКС, МИН)
    перебор, _ = explore(city, T, P, комбинации)
    сводка = explore_summary(перебор, ГРУППЫ)
    перебор = apply_filters(перебор, УСЛОВИЯ)
    export.table_csv(перебор, f'{OUT}/{КОД}_перебор.csv', index=False)
    export.table_csv(сводка, f'{OUT}/{КОД}_перебор_сводка.csv', index=False)
    print(f'Сочетаний: {len(комбинации)}; после условий: {len(перебор) - 1}')
    display(сводка[['показатель', 'от', 'до', 'сильнее всего зависит от доли']]) if 'display' in globals()         else print(сводка[['показатель', 'от', 'до', 'сильнее всего зависит от доли']].to_string(index=False))
else:
    print('Перебор сочетаний считает показатели правил v5.')

## Как повторить вручную в QGIS

| Шаг | В ноутбуке | В QGIS |
|---|---|---|
| 1. Перевод в метры | `mixuse.geometry.LocalProjection` | «Перепроецировать слой» → WGS 84 / UTM zone 36N (EPSG:32636) |
| 2. Группы сервисов | `таблица_групп_сервисов.csv` | присоединить таблицу групп к слою сервисов по полю типа («Свойства слоя» → «Связи»), затем поле `группа` |
| 3. Привязка к кварталам | `mixuse.geometry.inner_point`, `assign_points` | «Точка на поверхности» → «Объединить атрибуты по расположению (сводка)» с суммой или числом |
| 4. Поэтажная площадь, жители | `mixuse.blocks.table_from_layers` | «Калькулятор полей»: `building_area * storeys`, `living_area / 30` |
| 4. Категория нежилого здания | `mixuse.buildings.building_jobs_v5`: шаги 1–8 | «Подсчёт точек в полигоне» по зданиям для сервисов и социальных объектов; округлость — `4 * pi() * $area / ($perimeter ^ 2)`; затем CASE по зоне квартала, этажности и площади застройки; проверка — таблица `results/<код>_здания.csv` |
| 4. Встроенные помещения | `min(n · s0, 0,5 · S)` для жилого дома с сервисами К, ОД | «Подсчёт точек в полигоне» по жилым зданиям для К и ОД → «Калькулятор полей»: `min((n_К + n_ОД) * 100, 0.5 * building_area) * n_К / (n_К + n_ОД)` |
| 5. Окружение | `mixuse.compare.compare`: расстояние до границы ≤ R | выделить кварталы территории → «Буфер» R → «Центроиды» кварталов → «Выбрать по расположению» |
| 6. Доступ | расстояние ≤ r | «Буфер» r вокруг центроида жилого квартала → «Выбрать по расположению» кварталы с функцией |
| 6. Показатели | формулы в `compare` | «Статистика по полю» для сумм, формулы — в калькуляторе полей или Excel |